In [1]:
"""
FILE 3: BOOSTING MODELS
Run this on Colab Account 3
Trains: XGBoost, LightGBM, CatBoost, RandomForest, GradientBoosting
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import time

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                            confusion_matrix, balanced_accuracy_score)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from scipy.sparse import hstack, csr_matrix
from sklearn.utils.class_weight import compute_class_weight
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*100)
print("⚡ FILE 3: BOOSTING MODELS ⚡")
print("="*100)


⚡ FILE 3: BOOSTING MODELS ⚡


In [2]:
# ==================== DATA LOADING & PREPROCESSING ====================
print("\n📂 Loading data...")
df = pd.read_excel("bharatfakenewskosh.xlsx")

statement_col = 'Eng_Trans_Statement'
body_col = 'Eng_Trans_News_Body'
label_col = 'Label'

df = df[[statement_col, body_col, label_col]].dropna()

def clean_text_advanced(text):
    text = str(text).lower()
    if ':' in text:
        parts = text.split(':', 1)
        if any(word in parts[0] for word in ['fact-check', 'fact check', 'wrong', 'false']):
            text = parts[1]

    leak_words = ['fact-check', 'factcheck', 'debunked', 'hoax', 'busted', 'fake', 'false claim']
    for phrase in leak_words:
        text = text.replace(phrase, ' ')

    text = re.sub(r'http\S+|www\S+|@\w+|#\w+', '', text)
    text = re.sub(r'!{2,}', ' MULTIEXCLAIM ', text)
    text = re.sub(r'\?{2,}', ' MULTIQUESTION ', text)
    text = re.sub(r'\.{3,}', ' ELLIPSIS ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['statement_clean'] = df[statement_col].apply(clean_text_advanced)
df['body_clean'] = df[body_col].apply(clean_text_advanced)
df['combined_text'] = df['statement_clean'] + ' [SEP] ' + df['body_clean']

df['Label'] = df[label_col].map({'TRUE': 1, 'FALSE': 0}) if df[label_col].dtype == 'object' else df[label_col].astype(int)
df = df.dropna(subset=['Label'])
df['combined_words'] = df['combined_text'].str.split().str.len()
df = df[df['combined_words'] >= 15]
df = df.drop_duplicates(subset=['combined_text'], keep='first')

print(f"✅ Dataset: {len(df)} samples")


📂 Loading data...
✅ Dataset: 24455 samples


In [3]:
# ==================== FEATURE ENGINEERING ====================
print("\n🔧 Creating enhanced feature sets...")

def extract_advanced_features(df):
    features = pd.DataFrame()
    features['stmt_len'] = df['statement_clean'].str.len()
    features['body_len'] = df['body_clean'].str.len()
    features['combined_len'] = df['combined_text'].str.len()
    features['stmt_words'] = df['statement_clean'].str.split().str.len()
    features['body_words'] = df['body_clean'].str.split().str.len()
    features['total_words'] = features['stmt_words'] + features['body_words']
    features['stmt_body_ratio'] = features['stmt_len'] / (features['body_len'] + 1)
    features['word_density'] = features['total_words'] / (features['combined_len'] + 1)
    features['avg_word_len'] = features['combined_len'] / (features['total_words'] + 1)
    features['exclaim_count'] = df['combined_text'].str.count('!')
    features['question_count'] = df['combined_text'].str.count('\?')
    features['period_count'] = df['combined_text'].str.count('\.')
    features['comma_count'] = df['combined_text'].str.count(',')
    features['quote_count'] = df['combined_text'].str.count('"') + df['combined_text'].str.count("'")
    features['caps_count'] = df['combined_text'].str.count('[A-Z]')
    features['digit_count'] = df['combined_text'].str.count('\d')
    features['has_multiexclaim'] = df['combined_text'].str.contains('MULTIEXCLAIM').astype(int)
    features['has_multiquestion'] = df['combined_text'].str.contains('MULTIQUESTION').astype(int)
    features['has_ellipsis'] = df['combined_text'].str.contains('ELLIPSIS').astype(int)
    features['caps_ratio'] = features['caps_count'] / (features['combined_len'] + 1)
    features['digit_ratio'] = features['digit_count'] / (features['combined_len'] + 1)

    fake_keywords = ['viral', 'shocking', 'breaking', 'exposed', 'revealed',
                     'truth', 'must watch', 'exclusive', 'urgent', 'alert',
                     'proof', 'evidence', 'conspiracy', 'hidden', 'secret']
    for keyword in fake_keywords:
        features[f'has_{keyword}'] = df['combined_text'].str.contains(keyword, regex=False).astype(int)

    features['sentence_count'] = df['combined_text'].str.count(r'[.!?]') + 1
    features['avg_sentence_len'] = features['total_words'] / features['sentence_count']

    return features

linguistic_features = extract_advanced_features(df)
print(f"✅ Linguistic features: {linguistic_features.shape[1]} features")

X = df['combined_text']
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
ling_train, ling_test = train_test_split(linguistic_features, test_size=0.2, stratify=y, random_state=42)

print("\n🔄 Creating text vectorizers...")
vectorizer_tfidf_word = TfidfVectorizer(max_features=20000, ngram_range=(1,3),
                                        min_df=2, max_df=0.9, sublinear_tf=True)
X_train_tfidf_word = vectorizer_tfidf_word.fit_transform(X_train)
X_test_tfidf_word = vectorizer_tfidf_word.transform(X_test)

vectorizer_tfidf_char = TfidfVectorizer(max_features=8000, ngram_range=(2,6),
                                        analyzer='char', min_df=2, max_df=0.9)
X_train_tfidf_char = vectorizer_tfidf_char.fit_transform(X_train)
X_test_tfidf_char = vectorizer_tfidf_char.transform(X_test)

vectorizer_count = CountVectorizer(max_features=15000, ngram_range=(1,2),
                                   min_df=2, max_df=0.9, binary=True)
X_train_count = vectorizer_count.fit_transform(X_train)
X_test_count = vectorizer_count.transform(X_test)

scaler = StandardScaler()
ling_train_scaled = scaler.fit_transform(ling_train)
ling_test_scaled = scaler.transform(ling_test)

X_train_combined = hstack([X_train_tfidf_word, X_train_tfidf_char,
                           X_train_count, csr_matrix(ling_train_scaled)])
X_test_combined = hstack([X_test_tfidf_word, X_test_tfidf_char,
                          X_test_count, csr_matrix(ling_test_scaled)])

print(f"✅ Combined features: {X_train_combined.shape[1]} features")

print("\n📊 Applying SMOTE for class balancing...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_combined, y_train)
print(f"✅ After SMOTE: {X_train_resampled.shape[0]} samples")

class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
scale_pos_weight = class_weights[1]/class_weights[0]


🔧 Creating enhanced feature sets...
✅ Linguistic features: 38 features

🔄 Creating text vectorizers...
✅ Combined features: 43038 features

📊 Applying SMOTE for class balancing...
✅ After SMOTE: 23794 samples


In [4]:
# ==================== TRAIN BOOSTING MODELS ====================
print("\n" + "="*100)
print("⚡ TRAINING: BOOSTING MODELS")
print("="*100)

boosting_models = {
    "Random Forest [1000 est, depth=70]": RandomForestClassifier(
        n_estimators=1000, max_depth=70, min_samples_split=2,
        class_weight='balanced', n_jobs=-1, random_state=42
    ),
    "XGBoost [500 iter, depth=8]": XGBClassifier(
        n_estimators=500, max_depth=8, learning_rate=0.05,
        scale_pos_weight=scale_pos_weight, subsample=0.8, colsample_bytree=0.8,
        n_jobs=-1, random_state=42, eval_metric='logloss'
    ),
    "LightGBM [500 iter, depth=8]": LGBMClassifier(
        n_estimators=500, max_depth=8, learning_rate=0.05,
        class_weight='balanced', subsample=0.8, colsample_bytree=0.8,
        n_jobs=-1, random_state=42, verbose=-1
    ),
    "CatBoost [500 iter, depth=8]": CatBoostClassifier(
        iterations=500, depth=8, learning_rate=0.05,
        auto_class_weights='Balanced', verbose=0, random_state=42, thread_count=-1
    ),
    "Gradient Boosting [400 est]": GradientBoostingClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=8,
        subsample=0.8, random_state=42
    ),
}

results = {}
trained_models = {}

for name, model in boosting_models.items():
    print(f"\n⚡ Training: {name}")
    start_time = time.time()
    model.fit(X_train_resampled, y_train_resampled)
    train_time = time.time() - start_time

    preds = model.predict(X_test_combined)

    acc = accuracy_score(y_test, preds)
    bal_acc = balanced_accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)

    results[name] = {
        'Accuracy': acc, 'Balanced_Acc': bal_acc, 'Precision': prec,
        'Recall': rec, 'F1': f1, 'Time': train_time
    }
    trained_models[name] = model

    print(f"   ✅ Acc: {acc:.4f} ({acc*100:.2f}%) | F1: {f1:.4f} | Time: {train_time:.1f}s")



⚡ TRAINING: BOOSTING MODELS

⚡ Training: Random Forest [1000 est, depth=70]
   ✅ Acc: 0.6146 (61.46%) | F1: 0.7463 | Time: 139.5s

⚡ Training: XGBoost [500 iter, depth=8]
   ✅ Acc: 0.5809 (58.09%) | F1: 0.6688 | Time: 790.8s

⚡ Training: LightGBM [500 iter, depth=8]
   ✅ Acc: 0.6113 (61.13%) | F1: 0.7326 | Time: 179.5s

⚡ Training: CatBoost [500 iter, depth=8]
   ✅ Acc: 0.6119 (61.19%) | F1: 0.7439 | Time: 3529.8s

⚡ Training: Gradient Boosting [400 est]
   ✅ Acc: 0.6058 (60.58%) | F1: 0.7278 | Time: 6643.0s


In [5]:
# ==================== SAVE RESULTS ====================
results_df = pd.DataFrame(results).T
results_df.to_csv('results_part3_boosting.csv')
print("\n✅ Saved: results_part3_boosting.csv")

best_model_name = results_df.sort_values('F1', ascending=False).index[0]
joblib.dump(trained_models[best_model_name], 'best_model_part3.pkl')
print(f"✅ Saved best model: best_model_part3.pkl")

print("\n" + "="*100)
print("✅ FILE 3 COMPLETE!")
print("="*100)
print(f"Best Model: {best_model_name}")
print(f"Best Accuracy: {results[best_model_name]['Accuracy']:.4f}")


✅ Saved: results_part3_boosting.csv
✅ Saved best model: best_model_part3.pkl

✅ FILE 3 COMPLETE!
Best Model: Random Forest [1000 est, depth=70]
Best Accuracy: 0.6146
